# Approach A — Notebook 1: Target Prep & Pre-Computed Index
**Amazon ML Challenge 2026: Business Entity Resolution**

### Purpose:
1. Ingest 10.3M S2 + S3 target records (either raw TSVs or precomputed parquets).
2. Fit the 3 character $n$-gram TF-IDF vectorizers (`name_core`, `addr_norm`, `combined`).
3. Pre-transform and partition S23 sparse CSR matrices by country bucket (`US`, `India`, `France`, `__unknown__`).
4. Save the target bundle into `/kaggle/working/target_index/`.

**Output Artifacts:**
- `target_metadata.pkl`: S23 entity IDs, countries, raw text columns, fitted vectorizers.
- `target_sparse_matrices.pkl`: Pre-transformed sparse CSR matrices per country bucket.

*Publish `/kaggle/working/target_index/` as a Kaggle Dataset named `approach-a-target-index` for Notebooks 2A and 2B.*


In [ ]:
# 0. Dependencies Setup
!pip install sparse_dot_topn rapidfuzz lightgbm pyarrow --quiet


In [ ]:
# 1. Environment & Thread Budget Setup
import os, sys, time, glob, re, gc, pickle, unicodedata, warnings
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize

warnings.filterwarnings("ignore")

N_CORES = os.cpu_count() or 8
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, str(N_CORES))
os.environ.pop("GOMP_CPU_AFFINITY", None)
os.environ.pop("KMP_AFFINITY", None)

print(f"[Config] Detected CPU cores: {N_CORES}")


In [ ]:
# 2. Path Resolution (Auto-detects Kaggle vs Local)
def find_file(filename, search_root="/kaggle/input"):
    hits = glob.glob(os.path.join(search_root, "**", filename), recursive=True)
    return hits[0] if hits else None

if os.path.isdir("/kaggle/input"):
    TEST_DIR = os.path.dirname(find_file("test_source1.tsv") or "/kaggle/input/dataset/test/test_source1.tsv")
    WORK_DIR = "/kaggle/working"
else:
    TEST_DIR = os.path.abspath("data/student_resource/dataset/test")
    WORK_DIR = os.path.abspath("output/approach_a")

INDEX_DIR = os.path.join(WORK_DIR, "target_index")
os.makedirs(INDEX_DIR, exist_ok=True)
print("TEST_DIR:", TEST_DIR)
print("INDEX_DIR:", INDEX_DIR)


In [ ]:
# 3. Fast Text Normalization
LEGAL_FORMS = frozenset({
    "inc","incorporated","corp","corporation","co","company","ltd","limited",
    "llc","llp","plc","pvt","private","gmbh","ag","sa","sarl","srl","bv","nv",
    "dba","pte","pty","group","holdings","enterprises","ventures",
})

STREET_ABBR = {
    "rd":"road","st":"street","ave":"avenue","blvd":"boulevard","dr":"drive",
    "ln":"lane","ct":"court","pl":"place","cir":"circle","pkwy":"parkway",
    "hwy":"highway","trl":"trail","sq":"square","ter":"terrace","xing":"crossing",
    "nr":"near","opp":"opposite","flr":"floor","apt":"apartment",
}

US_STATES = {
    "al":"alabama","ak":"alaska","az":"arizona","ar":"arkansas","ca":"california",
    "co":"colorado","ct":"connecticut","de":"delaware","fl":"florida","ga":"georgia",
    "hi":"hawaii","id":"idaho","il":"illinois","in":"indiana","ia":"iowa",
    "ks":"kansas","ky":"kentucky","la":"louisiana","me":"maine","md":"maryland",
    "ma":"massachusetts","mi":"michigan","mn":"minnesota","ms":"mississippi",
    "mo":"missouri","mt":"montana","ne":"nebraska","nv":"nevada","nh":"new hampshire",
    "nj":"new jersey","nm":"new mexico","ny":"new york","nc":"north carolina",
    "nd":"north dakota","oh":"ohio","ok":"oklahoma","or":"oregon","pa":"pennsylvania",
    "ri":"rhode island","sc":"south carolina","sd":"south dakota","tn":"tennessee",
    "tx":"texas","ut":"utah","vt":"vermont","va":"virginia","wa":"washington",
    "wv":"west virginia","wi":"wisconsin","wy":"wyoming","dc":"district of columbia",
}

DEV_MAP = {
    'अ':'a','आ':'aa','इ':'i','ई':'ii','उ':'u','ऊ':'uu','ए':'e','ऐ':'ai','ओ':'o','औ':'au','ऋ':'ri',
    'क':'k','ख':'kh','ग':'g','घ':'gh','ङ':'ng','च':'ch','छ':'chh','ज':'j','झ':'jh','ञ':'ny',
    'ट':'t','ठ':'th','ड':'d','ढ':'dh','ण':'n','त':'t','थ':'th','द':'d','ध':'dh','न':'n',
    'प':'p','फ':'ph','ब':'b','भ':'bh','म':'m','य':'y','र':'r','ल':'l','व':'v','श':'sh','ष':'sh',
    'स':'s','ह':'h','ा':'aa','ि':'i','ी':'ii','ु':'u','ू':'uu','े':'e','ै':'ai','ो':'o','ौ':'au',
    'ं':'n','ः':'h','्':'','ृ':'ri','़':'','ँ':'n','ॉ':'o',
}

_punct_re = re.compile(r"[^a-z0-9\s]")
_ws_re    = re.compile(r"\s+")

def _translit(text):
    return "".join(DEV_MAP.get(ch, ch) for ch in text)

def _has_nonlatin(text):
    return any(ord(ch) > 127 and unicodedata.category(ch).startswith("L") for ch in text)

def _base(text):
    if not text or pd.isna(text): return ""
    text = str(text)
    if _has_nonlatin(text): text = _translit(text)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    return text.lower().strip()

def norm_name(text):
    t = _base(text)
    if not t: return "", "", ""
    t = t.replace("&", " and ")
    t = _punct_re.sub(" ", t)
    t = _ws_re.sub(" ", t).strip()
    name_full = t
    parts = re.split(r"\bdba\b", t)
    primary = parts[0].strip() if parts else t
    trade = parts[1].strip() if len(parts) > 1 else ""
    core = [tok for tok in primary.split() if tok not in LEGAL_FORMS]
    name_core = " ".join(core) if core else primary
    trade_core = " ".join(tok for tok in trade.split() if tok not in LEGAL_FORMS)
    return name_full, name_core, trade_core

def norm_addr(text):
    t = _base(text)
    if not t: return ""
    t = t.replace("&", " and ")
    t = re.sub(r"#+(\d)", r"\1", t)
    t = _punct_re.sub(" ", t)
    t = _ws_re.sub(" ", t).strip()
    out = [STREET_ABBR.get(tok, US_STATES.get(tok, tok)) for tok in t.split()]
    return " ".join(out)


In [ ]:
# 4. Load & Ingest Target S2 + S3
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp

parquet_hit = find_file("s23_norm.parquet") or find_file("s23_target.parquet")

if parquet_hit:
    print(f"Loading precomputed S23 target parquet: {parquet_hit}")
    s23 = pd.read_parquet(parquet_hit)
else:
    def _norm_chunk(texts, is_name=True):
        return [norm_name(t) for t in texts] if is_name else [norm_addr(t) for t in texts]

    def parallel_norm(series, is_name=True, n_workers=N_CORES):
        vals = series.tolist()
        n = len(vals)
        chunk_size = max(1, -(-n // n_workers))
        chunks = [vals[i:i + chunk_size] for i in range(0, n, chunk_size)]
        results = [None] * len(chunks)
        ctx = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else mp.get_context()
        with ProcessPoolExecutor(max_workers=n_workers, mp_context=ctx) as ex:
            futs = {ex.submit(_norm_chunk, c, is_name): i for i, c in enumerate(chunks)}
            for fut in as_completed(futs):
                results[futs[fut]] = fut.result()
        out = []
        for r in results: out.extend(r)
        return out

    def load_source_tsv(path):
        print(f"Reading {path} with pyarrow...")
        t0 = time.time()
        cols = ["entity_id", "business_name", "business_address", "country"]
        try:
            df = pd.read_csv(path, sep="\t", dtype=str, usecols=cols, engine="pyarrow").fillna("")
        except Exception:
            df = pd.read_csv(path, sep="\t", dtype=str, usecols=cols).fillna("")
        print(f"  Read {len(df):,} rows in {time.time()-t0:.1f}s. Normalizing...")
        t_n = time.time()
        name_res = parallel_norm(df["business_name"], is_name=True)
        df["name_full"]  = [r[0] for r in name_res]
        df["name_core"]  = [r[1] for r in name_res]
        df["name_trade"] = [r[2] for r in name_res]
        del name_res
        df["addr_norm"] = parallel_norm(df["business_address"], is_name=False)
        df["combined"] = df["name_core"] + " " + df["addr_norm"]
        df["country_norm"] = df["country"].map(lambda x: x.strip().lower() if x.strip() else "__unknown__")
        print(f"  Normalized {len(df):,} rows in {time.time()-t_n:.1f}s")
        return df[["entity_id","name_full","name_core","name_trade","addr_norm","combined","country","country_norm"]]

    s2 = load_source_tsv(os.path.join(TEST_DIR, "test_source2.tsv"))
    s3 = load_source_tsv(os.path.join(TEST_DIR, "test_source3.tsv"))
    s23 = pd.concat([s2, s3], ignore_index=True)
    del s2, s3; gc.collect()

print(f"Total S23 target pool: {len(s23):,} entities")


In [ ]:
# 5. Fit 3 Vectorizers & Pre-Transform Country Sparse Matrices
print("Fitting TF-IDF Vectorizers...")
t0 = time.time()

vec_name = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4), max_features=200_000, sublinear_tf=True, dtype=np.float32)
vec_name.fit(s23["name_core"])

vec_addr = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4), max_features=150_000, sublinear_tf=True, dtype=np.float32)
vec_addr.fit(s23["addr_norm"])

vec_comb = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4), max_features=200_000, sublinear_tf=True, dtype=np.float32)
vec_comb.fit(s23["combined"])
print(f"Fitted all 3 vectorizers in {time.time()-t0:.1f}s")

# Pre-transform sparse matrices per country bucket
print("Pre-transforming sparse country matrices for instant worker loading...")
t_sparse = time.time()

countries = sorted(set(s23["country_norm"].unique()))
print(f"Unique countries in S23: {countries}")

unknown_idx = np.where(s23["country_norm"].values == "__unknown__")[0].astype(np.int32)
sparse_country_dict = {}

for route_name, col, vec in [("name", "name_core", vec_name),
                             ("addr", "addr_norm", vec_addr),
                             ("comb", "combined", vec_comb)]:
    col_vals = s23[col].values
    unknown_mat = sk_normalize(vec.transform(col_vals[unknown_idx])) if len(unknown_idx) else None

    sparse_country_dict[route_name] = {"unknown_idx": unknown_idx, "buckets": {}}
    for c in countries:
        if c == "__unknown__":
            sparse_country_dict[route_name]["buckets"][c] = {
                "idx": np.arange(len(s23), dtype=np.int32),
                "mat_T": sk_normalize(vec.transform(col_vals)).T.tocsr()
            }
        else:
            exact_idx = np.where(s23["country_norm"].values == c)[0].astype(np.int32)
            if len(exact_idx) == 0 and len(unknown_idx) == 0: continue
            exact_mat = sk_normalize(vec.transform(col_vals[exact_idx])) if len(exact_idx) else None
            if exact_mat is not None and unknown_mat is not None:
                full_mat = sparse.vstack([exact_mat, unknown_mat], format="csr")
                full_idx = np.concatenate([exact_idx, unknown_idx])
                order = np.argsort(full_idx, kind="stable")
                full_idx = full_idx[order]
                full_mat = full_mat[order]
            elif exact_mat is not None:
                full_mat, full_idx = exact_mat, exact_idx
            else:
                full_mat, full_idx = unknown_mat, unknown_idx

            sparse_country_dict[route_name]["buckets"][c] = {
                "idx": full_idx,
                "mat_T": full_mat.T.tocsr()
            }

print(f"Pre-transformed country matrices in {time.time()-t_sparse:.1f}s")


In [ ]:
# 6. Save Artifacts for Worker Notebooks
meta = {
    "entity_ids": s23["entity_id"].values,
    "country_norm": s23["country_norm"].values,
    "country_raw": s23["country"].values,
    "name_core": s23["name_core"].values,
    "name_full": s23["name_full"].values,
    "name_trade": s23["name_trade"].values,
    "addr_norm": s23["addr_norm"].values,
    "combined": s23["combined"].values,
    "vec_name": vec_name,
    "vec_addr": vec_addr,
    "vec_comb": vec_comb,
}

meta_path = os.path.join(INDEX_DIR, "target_metadata.pkl")
matrices_path = os.path.join(INDEX_DIR, "target_sparse_matrices.pkl")

print(f"Saving {meta_path}...")
with open(meta_path, "wb") as f:
    pickle.dump(meta, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saving {matrices_path}...")
with open(matrices_path, "wb") as f:
    pickle.dump(sparse_country_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

print("="*60)
print(f"STAGE 1 COMPLETE: Target index artifacts successfully saved in {INDEX_DIR}")
print("Ready for Notebooks 2A and 2B!")
print("="*60)
